# CV Self-Learning Notebook: Image Pyramids and the Frequency Domain

This notebook is designed for self-study in computer vision.  
It combines **conceptual explanations**, **visual experiments**, and **hands-on tasks**.

## Learning Goals

By the end of this notebook, you should be able to:

- explain why naive downsampling causes aliasing
- understand the role of smoothing before subsampling
- build a Gaussian pyramid
- understand what information is lost in a Gaussian pyramid
- build and reconstruct a Laplacian pyramid
- interpret the Fourier transform of an image
- understand the difference between low-frequency and high-frequency image content
- implement simple frequency-domain filters
- connect image pyramids, filtering, and the Nyquist sampling theorem

---

## How to Use This Notebook

This notebook is written in a **task-based format**:

- **Concept**: key ideas you should understand
- **Task**: something you should implement, observe, or explain
- **Reflection**: short questions to check your understanding

Try to answer the reflection questions in your own words before moving on.

## 0. Background: Why This Topic Matters

Image pyramids and frequency-domain analysis are fundamental tools in computer vision.

They appear in many applications, such as:

- image compression
- multi-scale detection
- image blending
- denoising
- texture analysis
- feature extraction
- signal processing for vision models

A central theme of this notebook is that **images can be understood at multiple scales and in multiple representations**:

1. **Spatial domain**: the image as pixels arranged in 2D
2. **Frequency domain**: the image as a combination of low- and high-frequency patterns

Understanding both views helps build deeper intuition for vision algorithms.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

from numpy.fft import fft2, ifft2, fftshift, ifftshift

plt.rcParams["figure.figsize"] = (6, 6)
plt.rcParams["image.cmap"] = "gray"

## 1. Load an Image

You can replace the image path with your own image.  
For best results, use an image with textures, edges, and repeated patterns.

In [ ]:
img_path = "test.jpg"  # Replace with your own image path if needed
img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

if img is None:
    # Fallback synthetic image if no file is provided
    img = np.zeros((256, 256), dtype=np.uint8)
    cv2.rectangle(img, (30, 30), (110, 110), 180, -1)
    cv2.circle(img, (180, 80), 35, 220, -1)
    for i in range(0, 256, 8):
        img[:, i:i+2] = 255
    cv2.putText(img, "CV", (70, 210), cv2.FONT_HERSHEY_SIMPLEX, 2, 200, 3, cv2.LINE_AA)

plt.imshow(img)
plt.title("Input Image")
plt.axis("off")
plt.show()

## 2. Naive Downsampling

### Concept

A simple way to reduce image size by a factor of 2 is to keep every other row and every other column.
This operation is often called **naive downsampling** (or decimation without pre-filtering).

More concretely, if the original image is $I[x, y]$, naive downsampling produces

$$
I_{\downarrow 2}[x, y] = I[2x, 2y]
$$

This is computationally cheap and easy to implement, but it ignores a key signal-processing principle:

- A digital image is a sampled version of an underlying continuous scene.
- Downsampling reduces the sampling rate.
- Before reducing the sampling rate, frequencies above the new Nyquist limit should be removed.

If we skip that low-pass filtering step, high-frequency details (fine textures, sharp edges, thin stripes) are folded into lower frequencies after sampling. This is **aliasing**.

Common visual artifacts caused by aliasing include:

- jagged or stair-step edges,
- moire patterns on repetitive textures (fabric, fences, tiles),
- false low-frequency patterns that were not present in the original image.

So, naive downsampling is useful as a baseline to demonstrate what goes wrong without anti-aliasing. In practice, we usually apply a blur (low-pass filter) before subsampling.

### Task 2.1
Implement naive downsampling by a factor of 2.

In [ ]:
def downsample_naive(image):
    return image[::2, ::2]

img_naive = downsample_naive(img)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(img_naive)
plt.title("Naively Downsampled")
plt.axis("off")
plt.show()

### Reflection

1. What kinds of visual detail are damaged by naive downsampling?
2. Do edges become smoother or more jagged?
3. If the image contains repeated fine textures, what do you expect to happen?

## 3. Aliasing in 1D

### Concept

Aliasing is easier to understand first in one dimension.

Suppose a signal oscillates very quickly, but we only observe a small number of samples.  
Then the sampled points may look like they came from a slower signal.

This is the key surprise of aliasing:

> **Undersampling can make a high-frequency signal appear to be low-frequency.**

### Task 3.1
Visualize a high-frequency sine wave and a sparse set of sampled points.

In [ ]:
x = np.linspace(0, 4 * np.pi, 1000)
y = np.sin(18 * x)

sample_step = 30
x_sampled = x[::sample_step]
y_sampled = y[::sample_step]

plt.figure(figsize=(10, 4))
plt.plot(x, y, label="Original high-frequency signal")
plt.scatter(x_sampled, y_sampled, color="red", label="Sparse samples")
plt.legend()
plt.title("Aliasing in 1D")
plt.show()

### Reflection

1. If you only looked at the red points, would you correctly guess the original signal frequency?
2. Why does undersampling lose more than just detail?
3. How is this idea connected to jagged edges or moiré patterns in images?

## 4. Anti-Aliasing: Smooth Before You Sample

### Concept

To reduce aliasing, we usually apply a **low-pass filter** before downsampling.

A low-pass filter removes high-frequency components that cannot be safely represented at the lower resolution.

A Gaussian blur is a common choice because it smooths the image gently and avoids strong ringing artifacts.

This gives us a better downsampling pipeline:

1. blur the image
2. subsample the image

### Task 4.1
Implement Gaussian downsampling.

In [ ]:
def downsample_gaussian(image, ksize=5, sigma=1.0):
    blurred = cv2.GaussianBlur(image, (ksize, ksize), sigma)
    return blurred[::2, ::2]

img_gaussian = downsample_gaussian(img)

plt.figure(figsize=(15, 4))
plt.subplot(1, 3, 1)
plt.imshow(img)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(img_naive)
plt.title("Naive downsampling")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(img_gaussian)
plt.title("Gaussian + downsampling")
plt.axis("off")
plt.show()

### Reflection

1. Why does smoothing help before downsampling?
2. Why does anti-aliasing still lose information?
3. Why is this loss usually preferable to aliasing artifacts?

## 5. Gaussian Pyramid

### Concept

A **Gaussian pyramid** is a sequence of images at progressively lower resolutions.

Each level is produced by:

1. smoothing the previous level
2. downsampling it

As we move upward in the pyramid:

- image size decreases
- fine details disappear
- large structures remain

This is called a **multi-scale representation**.

### Task 5.1
Build a Gaussian pyramid with several levels.

In [ ]:
def build_gaussian_pyramid(image, levels=4, ksize=5, sigma=1.0):
    pyramid = [image.astype(np.float32)]
    current = image.astype(np.float32)

    for _ in range(levels):
        current = cv2.GaussianBlur(current, (ksize, ksize), sigma)
        current = current[::2, ::2]
        pyramid.append(current)

    return pyramid

gaussian_pyramid = build_gaussian_pyramid(img, levels=4)

plt.figure(figsize=(16, 4))
for i, level_img in enumerate(gaussian_pyramid):
    plt.subplot(1, len(gaussian_pyramid), i + 1)
    plt.imshow(level_img)
    plt.title(f"Level {i}\n{level_img.shape[1]}x{level_img.shape[0]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

### Reflection

1. Which information survives in higher pyramid levels?
2. Which information disappears first?
3. Why might multi-scale representations be useful for detection or recognition?

## 6. Why Gaussian Pyramids Are Lossy

### Concept

A Gaussian pyramid is useful, but it is **not directly reversible**.

Why?

Because each stage performs blurring and subsampling, both of which discard information.  
Once high-frequency information is removed, it cannot be perfectly recovered from the smaller image alone.

A helpful idea is to compare:

- the original image
- a blurred-and-resized approximation of it

Their difference tells us what was lost.

### Task 6.1
Upsample a coarser Gaussian level and compare it with the finer level.

In [ ]:
level0 = gaussian_pyramid[0]
level1 = gaussian_pyramid[1]
level1_up = cv2.resize(level1, (level0.shape[1], level0.shape[0]), interpolation=cv2.INTER_LINEAR)
residual = level0 - level1_up

plt.figure(figsize=(15, 4))
plt.subplot(1, 3, 1)
plt.imshow(level0)
plt.title("Level 0")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(level1_up)
plt.title("Upsampled Level 1")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(residual)
plt.title("Residual = Level 0 - Upsampled Level 1")
plt.axis("off")
plt.show()

### Reflection

1. What kinds of structures appear in the residual image?
2. Why do edges often stand out in the residual?
3. What does this suggest about what the coarser Gaussian image fails to preserve?

## 7. Laplacian Pyramid

### Concept

A **Laplacian pyramid** stores the information lost between adjacent Gaussian pyramid levels.

At each level, we keep:

- a coarse approximation from the Gaussian pyramid
- a residual image that captures missing detail

This makes the representation much more informative than storing only blurred images.

A Laplacian pyramid is typically built from a Gaussian pyramid.

### Task 7.1
Build a Laplacian pyramid.

In [ ]:
def build_laplacian_pyramid(image, levels=4, ksize=5, sigma=1.0):
    g_pyr = build_gaussian_pyramid(image, levels=levels, ksize=ksize, sigma=sigma)
    l_pyr = []

    for i in range(len(g_pyr) - 1):
        current = g_pyr[i]
        next_level = g_pyr[i + 1]
        upsampled = cv2.resize(next_level, (current.shape[1], current.shape[0]), interpolation=cv2.INTER_LINEAR)
        lap = current - upsampled
        l_pyr.append(lap)

    l_pyr.append(g_pyr[-1])  # smallest Gaussian image
    return g_pyr, l_pyr

gaussian_pyramid, laplacian_pyramid = build_laplacian_pyramid(img, levels=4)

plt.figure(figsize=(16, 4))
for i, level_img in enumerate(laplacian_pyramid):
    plt.subplot(1, len(laplacian_pyramid), i + 1)
    plt.imshow(level_img)
    plt.title(f"Laplacian {i}" if i < len(laplacian_pyramid)-1 else "Smallest Gaussian")
    plt.axis("off")
plt.tight_layout()
plt.show()

### Concept Check

A Laplacian pyramid stores:

- the residuals at each level
- the smallest Gaussian image at the top

This is enough to reconstruct the original image.

### Task 7.2
Reconstruct the original image from the Laplacian pyramid.

In [ ]:
def reconstruct_from_laplacian_pyramid(l_pyr):
    current = l_pyr[-1]
    for i in reversed(range(len(l_pyr) - 1)):
        current = cv2.resize(current, (l_pyr[i].shape[1], l_pyr[i].shape[0]), interpolation=cv2.INTER_LINEAR)
        current = current + l_pyr[i]
    return current

reconstructed = reconstruct_from_laplacian_pyramid(laplacian_pyramid)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(img)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(reconstructed)
plt.title("Reconstructed")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(np.abs(img.astype(np.float32) - reconstructed))
plt.title("Absolute Error")
plt.axis("off")
plt.show()

print("Mean absolute reconstruction error:", np.mean(np.abs(img.astype(np.float32) - reconstructed)))

### Reflection

1. Why can a Laplacian pyramid be reconstructed, while a Gaussian pyramid alone cannot?
2. Why do Laplacian levels often look edge-like?
3. How is this related to the idea of local detail or high-frequency content?

## 8. From Spatial Domain to Frequency Domain

### Concept

In the **spatial domain**, we look at pixel intensities directly.

In the **frequency domain**, we describe the image as a combination of patterns that vary at different rates:

- **low frequencies**: slow variation, smooth intensity changes, large structures
- **high frequencies**: rapid variation, edges, fine textures, noise

The Fourier transform converts an image from spatial domain to frequency domain.

### Task 8.1
Compute and visualize the Fourier magnitude spectrum.

In [ ]:
def fft_magnitude_spectrum(image):
    f = fft2(image.astype(np.float32))
    fshift = fftshift(f)
    magnitude = np.log(np.abs(fshift) + 1)
    return fshift, magnitude

fshift, magnitude = fft_magnitude_spectrum(img)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title("Spatial Domain")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(magnitude)
plt.title("Log Magnitude Spectrum")
plt.axis("off")
plt.show()

### Concept Check

After `fftshift`, the **center of the spectrum** corresponds to low frequencies.  
As you move away from the center, frequencies become higher.

### Reflection

1. Why is the center bright for many natural images?
2. What do strong high-frequency responses mean visually?
3. If an image is very smooth, what should its spectrum look like?

## 9. Low-Pass and High-Pass Filtering in the Frequency Domain

### Concept

Filtering can be done in two equivalent ways:

- convolution in the spatial domain
- multiplication in the frequency domain

This is one of the most important ideas in signal processing.

A **low-pass filter** keeps low frequencies and removes high frequencies.  
A **high-pass filter** removes low frequencies and keeps rapid changes.

### Task 9.1
Implement a simple circular low-pass filter mask.

In [ ]:
def low_pass_filter_frequency(image, radius=30):
    image = image.astype(np.float32)
    f = fft2(image)
    fshift = fftshift(f)

    h, w = image.shape
    cy, cx = h // 2, w // 2
    Y, X = np.ogrid[:h, :w]
    dist2 = (Y - cy) ** 2 + (X - cx) ** 2
    mask = (dist2 <= radius ** 2).astype(np.float32)

    filtered_shift = fshift * mask
    filtered = np.real(ifft2(ifftshift(filtered_shift)))
    return filtered, mask

low_img, low_mask = low_pass_filter_frequency(img, radius=25)

plt.figure(figsize=(15, 4))
plt.subplot(1, 3, 1)
plt.imshow(low_mask)
plt.title("Low-pass Mask")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(img)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(low_img)
plt.title("Low-pass Result")
plt.axis("off")
plt.show()

### Task 9.2
Implement a high-pass filter by removing the low-frequency center.

In [ ]:
def high_pass_filter_frequency(image, radius=25):
    image = image.astype(np.float32)
    f = fft2(image)
    fshift = fftshift(f)

    h, w = image.shape
    cy, cx = h // 2, w // 2
    Y, X = np.ogrid[:h, :w]
    dist2 = (Y - cy) ** 2 + (X - cx) ** 2
    mask = (dist2 > radius ** 2).astype(np.float32)

    filtered_shift = fshift * mask
    filtered = np.real(ifft2(ifftshift(filtered_shift)))
    return filtered, mask

high_img, high_mask = high_pass_filter_frequency(img, radius=25)

plt.figure(figsize=(15, 4))
plt.subplot(1, 3, 1)
plt.imshow(high_mask)
plt.title("High-pass Mask")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(img)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(high_img)
plt.title("High-pass Result")
plt.axis("off")
plt.show()

### Reflection

1. Why does the low-pass result look blurry?
2. Why does the high-pass result emphasize edges and fine texture?
3. Which result is more sensitive to noise?

## 10. Convolution Theorem

### Concept

The **convolution theorem** states:

> Convolution in the spatial domain corresponds to multiplication in the frequency domain.

This means that applying a linear filter in image space can also be understood as modifying the image spectrum.

This idea helps explain why smoothing removes high frequencies and why edge-like operations emphasize them.

### Task 10.1
Compare spatial Gaussian blur and frequency-domain low-pass filtering.

In [ ]:
blur_spatial = cv2.GaussianBlur(img.astype(np.float32), (11, 11), 2.0)
blur_freq, _ = low_pass_filter_frequency(img, radius=20)

plt.figure(figsize=(15, 4))
plt.subplot(1, 3, 1)
plt.imshow(img)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(blur_spatial)
plt.title("Spatial Gaussian Blur")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(blur_freq)
plt.title("Frequency Low-pass")
plt.axis("off")
plt.show()

### Reflection

1. Are the two filtered results identical?
2. If not, why not?
3. What properties must match for them to become closer?

## 11. Nyquist Sampling Theorem

### Concept

The Nyquist sampling theorem tells us how densely we must sample a signal to avoid aliasing.

The key message is:

> To represent a signal without aliasing, the sampling frequency must be at least twice the highest frequency present in the signal.

For images, this means:

- high-frequency content must be removed before reducing resolution
- otherwise, the lower-resolution image cannot represent the original structure correctly

This is exactly why Gaussian pyramids smooth before subsampling.

### Task 11.1
Experiment with different downsampling strategies and blur strengths.

In [ ]:
sigmas = [0.0, 0.8, 1.5, 3.0]
plt.figure(figsize=(14, 8))

for i, s in enumerate(sigmas, 1):
    if s == 0.0:
        result = img[::2, ::2]
        title = "No blur"
    else:
        result = cv2.GaussianBlur(img, (7, 7), s)[::2, ::2]
        title = f"Gaussian sigma={s}"
    plt.subplot(2, 2, i)
    plt.imshow(result)
    plt.title(title)
    plt.axis("off")

plt.tight_layout()
plt.show()

### Reflection

1. What happens when the blur is too weak before downsampling?
2. What happens when the blur is very strong?
3. How does this illustrate the trade-off between preserving detail and preventing aliasing?

## 12. Mini Project: Build a Hybrid Image

### Concept

A hybrid image combines:

- low-frequency content from one image
- high-frequency content from another image

Up close, one image may dominate.  
From far away or at smaller scale, the other image may become more noticeable.

This is a powerful demonstration of frequency-based perception.

### Task 12.1
Use two images to construct a hybrid image.

You can replace the placeholders below with your own image pair.

In [ ]:
img1 = img.astype(np.float32)

# Use a shifted version of the same image as a placeholder if no second image is available
img2 = np.roll(img.astype(np.float32), shift=20, axis=1)

low_part, _ = low_pass_filter_frequency(img1, radius=20)
high_part, _ = high_pass_filter_frequency(img2, radius=20)

hybrid = np.clip(low_part + high_part, 0, 255)

plt.figure(figsize=(15, 4))
plt.subplot(1, 3, 1)
plt.imshow(low_part)
plt.title("Low-frequency Part")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(high_part)
plt.title("High-frequency Part")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(hybrid)
plt.title("Hybrid Image")
plt.axis("off")
plt.show()

### Reflection

1. Why does viewing distance affect hybrid image perception?
2. Which part dominates at fine scale?
3. Which part dominates at coarse scale?

## 13. Summary

In this notebook, you studied several key ideas in computer vision and signal processing:

- **Naive downsampling** causes aliasing
- **Anti-aliasing** smooths the signal before subsampling
- A **Gaussian pyramid** stores progressively coarser image structure
- A **Laplacian pyramid** stores residual details and supports reconstruction
- The **Fourier transform** reveals image content in terms of frequency
- **Low-pass filters** preserve smooth structure
- **High-pass filters** emphasize edges and detail
- The **Nyquist theorem** explains when aliasing occurs

These ideas are foundational for later topics in computer vision, including image blending, scale-space theory, feature extraction, and multi-scale deep learning intuition.

## 14. Optional Extensions

If you want to continue exploring, try these extensions:

1. Replace the simple circular frequency mask with a Gaussian mask.
2. Compare box blur and Gaussian blur in both spatial and frequency domains.
3. Build a color-image version of the Gaussian and Laplacian pyramids.
4. Use real image pairs for hybrid images.
5. Study how repeated textures produce moiré artifacts after downsampling.
6. Try to estimate how pyramid representations change the storage cost of an image.

## 15. Short Written Questions

Write brief answers to the following:

1. What is aliasing, in your own words?
2. Why is smoothing necessary before downsampling?
3. What is the main difference between a Gaussian pyramid and a Laplacian pyramid?
4. In the Fourier spectrum, what does the center represent?
5. Why are edges usually associated with high-frequency content?
6. How does the Nyquist theorem connect to image pyramids?